### Libraries 

In [8]:
import requests

### URL to use to search via search engine

In [9]:
INDEX_PAGE = "https://idr-testing.openmicroscopy.org/webclient/?experimenter=-1"
SEARCH_ENGINE_URL = "https://idr-testing.openmicroscopy.org/searchengine/api/v1/resources/{type}/"
KEY_VALUE_SEARCH = SEARCH_ENGINE_URL + "search/?key={key}&value={value}"
KEYS_SEARCH = SEARCH_ENGINE_URL + "container_keyvalues/?key={key}&container_name={name}"

### Parameters

In [10]:
# Key used by search engine
KEY = "Gene Identifier"

In [11]:
# create http session
with requests.Session() as session:
    request = requests.Request('GET', INDEX_PAGE)
    prepped = session.prepare_request(request)
    response = session.send(prepped)
    if response.status_code != 200:
        response.raise_for_status()

In [12]:
# Helper method to load the possible values for a given key
# name: study name e.g. idr0101
def load_values_for_given_key(name):
    values = []
    qs1 = {'type': 'image', 'name' : name, 'key': KEY}
    url = KEYS_SEARCH.format(**qs1)  
    json = session.get(url).json()
    for d in json:
        if d['results']:
            for r in d['results']:
                values.append(r['value'])
    return values

### Load value

Loads the values associated to the specified KEY for two studies and sorts the returned values

In [14]:
values = load_values_for_given_key('idr0070')
values.extend(load_values_for_given_key('idr0114'))
values.sort()

In [16]:
print(len(values))

144


### Functional enrichment
We use g:Profiler to find the GO terms associated to the values
In the first instance we look at the Biological Process i.e. ``GO:BP`` as the source.

In [20]:
from gprofiler import GProfiler

gp = GProfiler(return_dataframe=True)
data = gp.profile(organism='hsapiens',
            query=values, sources=['GO:BP'])
display(data)

,source,native,name,p_value,significant,description,term_size,query_size,intersection_size,effective_domain_size,precision,recall,query,parents
0,GO:BP,GO:0048731,system development,9.240037e-44,True,"""The process whose specific outcome is the pro...",4369,129,103,21092,0.798450,0.023575,query_1,"[GO:0007275, GO:0048856]"
1,GO:BP,GO:0007275,multicellular organism development,6.742310e-42,True,"""The biological process whose specific outcome...",4823,129,105,21092,0.813953,0.021771,query_1,"[GO:0032501, GO:0048856]"
2,GO:BP,GO:0007417,central nervous system development,3.983838e-41,True,"""The process whose specific outcome is the pro...",1065,129,61,21092,0.472868,0.057277,query_1,"[GO:0007399, GO:0048731]"
3,GO:BP,GO:0048856,anatomical structure development,2.263023e-39,True,"""The biological process whose specific outcome...",5836,129,110,21092,0.852713,0.018849,query_1,[GO:0032502]
4,GO:BP,GO:0009653,anatomical structure morphogenesis,4.767987e-39,True,"""The process in which anatomical structures ar...",2722,129,83,21092,0.643411,0.030492,query_1,"[GO:0032502, GO:0048856]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
619,GO:BP,GO:0072203,cell proliferation involved in metanephros dev...,4.641547e-02,True,"""The multiplication or reproduction of cells, ...",10,129,3,21092,0.023256,0.300000,query_1,"[GO:0001656, GO:0072111]"
620,GO:BP,GO:0021520,spinal cord motor neuron cell fate specification,4.641547e-02,True,"""The process in which a cell becomes capable o...",10,129,3,21092,0.023256,0.300000,query_1,"[GO:0021522, GO:0048665]"
621,GO:BP,GO:0048664,neuron fate determination,4.641547e-02,True,"""The process in which a cell becomes capable o...",10,129,3,21092,0.023256,0.300000,query_1,"[GO:0001709, GO:0048663]"
622,GO:BP,GO:0060513,prostatic bud formation,4.641547e-02,True,"""The morphogenetic process in which a region o...",10,129,3,21092,0.023256,0.300000,query_1,"[GO:0016331, GO:0048645, GO:0060572, GO:006060..."


In [26]:
# Export the "native" colum to CSV
cols = [1]
df = data[data.columns[cols]]
df.to_csv("go_bp.csv", index=False, header=False)

In [18]:
# Check orthologs
gp.orth(organism='hsapiens',
            query=values,
            target='mmusculus')

,incoming,converted,ortholog_ensg,n_incoming,n_converted,n_result,name,description,namespaces
0,ENSG00000007372,ENSG00000007372,ENSMUSG00000027168,1,1,1,Pax6,paired box 6 [Source:MGI Symbol;Acc:MGI:97490],"ARRAYEXPRESS,ENSG"
1,ENSG00000007372,ENSG00000007372,ENSMUSG00000027168,2,1,1,Pax6,paired box 6 [Source:MGI Symbol;Acc:MGI:97490],"ARRAYEXPRESS,ENSG"
2,ENSG00000009709,ENSG00000009709,ENSMUSG00000028736,3,1,1,Pax7,paired box 7 [Source:MGI Symbol;Acc:MGI:97491],"ARRAYEXPRESS,ENSG"
3,ENSG00000012048,ENSG00000012048,ENSMUSG00000017146,4,1,1,Brca1,"breast cancer 1, early onset [Source:MGI Symbo...","ARRAYEXPRESS,ENSG"
4,ENSG00000016082,ENSG00000016082,ENSMUSG00000042258,5,1,1,Isl1,"ISL1 transcription factor, LIM/homeodomain [So...","ARRAYEXPRESS,ENSG"
...,...,...,...,...,...,...,...,...,...
140,ENSG00000254647,ENSG00000254647,ENSMUSG00000000215,141,1,1,Ins2,insulin II [Source:MGI Symbol;Acc:MGI:96573],"ARRAYEXPRESS,ENSG"
141,ENSG00000254647,ENSG00000254647,ENSMUSG00000035804,141,1,2,Ins1,insulin I [Source:MGI Symbol;Acc:MGI:96572],"ARRAYEXPRESS,ENSG"
142,ENSG00000258947,ENSG00000258947,ENSMUSG00000062380,142,1,1,Tubb3,"tubulin, beta 3 class III [Source:MGI Symbol;A...","ARRAYEXPRESS,ENSG"
143,ENSG00000258947,ENSG00000258947,ENSMUSG00000062380,143,1,1,Tubb3,"tubulin, beta 3 class III [Source:MGI Symbol;A...","ARRAYEXPRESS,ENSG"
